In [1]:
import pandas as pd
import numpy as np
import pygeohash as pgh
import matplotlib.pyplot as plt
import geohash2
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
train_df = pd.read_csv("../data/raw/train.csv")
test_df = pd.read_csv("../data/raw/test.csv")

print(train_df.shape)
print(test_df.shape)

train_df.head()

(77299, 11)
(41778, 10)


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [3]:
train_df.columns

Index(['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType',
       'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature',
       'Weather'],
      dtype='str')

In [4]:
for col in train_df.columns:
    # print(col)
    # print(train_df[col].unique()[:5])
    print(col,": ",train_df[col].dtype)

Index :  int64
geohash :  str
day :  int64
timestamp :  str
demand :  float64
RoadType :  str
NumberofLanes :  int64
LargeVehicles :  str
Landmarks :  str
Temperature :  float64
Weather :  str


# Data cleaning

In [5]:
def convert_string_columns_to_lowercase(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert all string/object columns to lowercase.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe

    Returns
    -------
    pd.DataFrame
        Dataframe with lowercase string values
    """
    df = df.copy()

    str_cols = df.select_dtypes(include=["object"]).columns

    for col in str_cols:
        df[col] = df[col].astype(str).str.lower()

    return df


def fill_categorical_missing_values(
    df: pd.DataFrame,
    categorical_cols: list
) -> pd.DataFrame:
    """
    Replace NaN and 'nan' strings in categorical columns
    with the most frequent value (mode).

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe
    categorical_cols : list
        List of categorical columns

    Returns
    -------
    pd.DataFrame
        Updated dataframe
    """
    df = df.copy()

    for col in categorical_cols:

        # Convert string 'nan' back to actual NaN
        df[col] = df[col].replace("nan", np.nan)

        # Most frequent value
        mode_value = df[col].mode()[0]

        # Fill missing values
        df[col] = df[col].fillna(mode_value)

    return df


def fill_temperature_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Replace missing Temperature values with mean temperature.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
    """
    df = df.copy()

    mean_temp = df["Temperature"].mean()

    df["Temperature"] = df["Temperature"].fillna(mean_temp)

    return df



In [6]:

def clean_dataset(org_df: pd.DataFrame) -> pd.DataFrame:
    """
    Complete data cleaning pipeline.

    Steps:
    1. Convert all strings to lowercase
    2. Fill missing RoadType with mode
    3. Fill missing Weather with mode
    4. Fill missing Temperature with mean

    Parameters
    ----------
    org_df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
    """
    df = org_df.copy()
    try:
        df = convert_string_columns_to_lowercase(df)

        df = fill_categorical_missing_values(
            df,
            categorical_cols=["RoadType", "Weather"]
        )

        df = fill_temperature_missing_values(df)

        return df

    except Exception as e:
        print(f"Error occurred while cleaning dataset: {e}")
        return org_df


# Feature engineering

In [7]:
def add_lat_lon(df: pd.DataFrame) -> pd.DataFrame:
    """
    Decode geohash into latitude and longitude.
    """
    df = df.copy()

    decoded = df["geohash"].apply(geohash2.decode)

    df["lat"] = decoded.apply(lambda x: x[0])
    df["lon"] = decoded.apply(lambda x: x[1])

    return df


def add_hour_feature(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extract hour from timestamp column.

    Example:
    '13:45' -> 13
    """
    df = df.copy()

    df["hour"] = (
        df["timestamp"]
        .astype(str)
        .str.split(":")
        .str[0]
        .astype(int)
    )

    return df


def add_geo_hour_feature(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create geohash-hour interaction feature.
    """
    df = df.copy()

    df["geo_hour"] = (
        df["geohash"].astype(str)
        + "_"
        + df["hour"].astype(str)
    )

    return df


def get_period(hour: int) -> str:
    """
    Map hour to time period.
    """

    if hour < 5:
        return "night"

    elif hour < 12:
        return "morning"

    elif hour < 17:
        return "afternoon"

    elif hour < 21:
        return "evening"

    else:
        return "late_night"


def add_period_feature(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create period feature from hour.
    """
    df = df.copy()

    df["period"] = df["hour"].apply(get_period)

    return df


def add_geo_period_feature(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create geohash-period interaction feature.
    """
    df = df.copy()

    df["geo_period"] = (
        df["geohash"].astype(str)
        + "_"
        + df["period"].astype(str)
    )

    return df


def add_roadtype_period_feature(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create RoadType-period interaction feature.
    """
    df = df.copy()

    df["roadtype_period"] = (
        df["RoadType"].astype(str)
        + "_"
        + df["period"].astype(str)
    )

    return df

def add_cyclical_hour_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add cyclical encoding for hour using sine and cosine.
    Assumes hour is in [0, 23].
    """
    df = df.copy()

    df["sin_hour"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["cos_hour"] = np.cos(2 * np.pi * df["hour"] / 24)

    return df

def add_cyclical_day_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add cyclical encoding for day using sine and cosine.
    Assumes day is in [0, 6].
    """
    df = df.copy()

    df["sin_day"] = np.sin(2 * np.pi * df["day"] / 7)
    df["cos_day"] = np.cos(2 * np.pi * df["day"] / 7)

    return df

def add_squared_temperature(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add squared temperature feature.
    """
    df = df.copy()

    df["temperature_sq"] = df["Temperature"] ** 2 - df["Temperature"]

    return df

def add_hour_squared(df):
    df = df.copy()
    df["hour_sq"] = df["hour"] ** 2
    return df

def add_rush_hour_flag(df):
    df = df.copy()

    df["is_morning_peak"] = (
        df["hour"].between(7, 9)
    ).astype(int)

    df["is_evening_peak"] = (
        df["hour"].between(17, 20)
    ).astype(int)

    return df

def add_spatial_interaction(df):
    df = df.copy()

    lat = df["lat"].astype(float)
    lon = df["lon"].astype(float)

    df["lat_lon_interaction"] = lat * lon

    return df

def add_temp_bucket(df):
    df = df.copy()

    df["temp_bucket"] = pd.cut(
        df["Temperature"],
        bins=[-100, 15, 25, 35, 100],
        labels=["cold", "mild", "warm", "hot"]
    )

    return df

def add_weather_temp(df):
    df = df.copy()

    df["weather_temp"] = (
        df["Weather"].astype(str)
        + "_"
        + pd.cut(
            df["Temperature"],
            bins=[-100,15,25,35,100],
            labels=["cold","mild","warm","hot"]
        ).astype(str)
    )

    return df

def add_geo_weather(df):
    df = df.copy()

    df["geo_weather"] = (
        df["geohash"].astype(str)
        + "_"
        + df["Weather"].astype(str)
    )

    return df

def add_geo_road(df):
    df = df.copy()

    df["geo_road"] = (
        df["geohash"].astype(str)
        + "_"
        + df["RoadType"].astype(str)
    )

    return df

def add_geo_temp_bucket(df):
    df = df.copy()

    bucket = pd.cut(
        df["Temperature"],
        bins=[-100,15,25,35,100],
        labels=["cold","mild","warm","hot"]
    )

    df["geo_temp"] = (
        df["geohash"].astype(str)
        + "_"
        + bucket.astype(str)
    )

    return df


In [8]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Complete feature engineering pipeline.

    Features created:
    - lat
    - lon
    - hour
    - geo_hour
    - period
    - geo_period
    - roadtype_period
    - sin_hour
    - cos_hour
    - temperature_sq
    - hour_sq
    - is_morning_peak
    - is_evening_peak
    - lat_lon_interaction
    - temp_bucket
    - weather_temp
    - geo_weather
    - geo_road
    - geo_temp
    """
    df = add_lat_lon(df)

    df = add_hour_feature(df)

    df = add_geo_hour_feature(df)

    df = add_period_feature(df)

    df = add_geo_period_feature(df)

    df = add_roadtype_period_feature(df)

    df = add_cyclical_hour_features(df)

    df = add_squared_temperature(df)

    df = add_hour_squared(df)

    df = add_rush_hour_flag(df)

    df = add_spatial_interaction(df)

    df = add_temp_bucket(df)

    df = add_weather_temp(df)

    df = add_geo_weather(df)

    df = add_geo_road(df)

    df = add_geo_temp_bucket(df)

    return df

In [9]:
train_df = clean_dataset(train_df)

/tmp/ipykernel_15764/3791486193.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df.select_dtypes(include=["object"]).columns


In [10]:
train_df = engineer_features(train_df)

In [11]:
train_df.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,...,temperature_sq,hour_sq,is_morning_peak,is_evening_peak,lat_lon_interaction,temp_bucket,weather_temp,geo_weather,geo_road,geo_temp
0,0,qp02z1,48,0:0,0.048804,residential,1,not allowed,no,16.405354,...,252.730296,0,0,0,-497.036,mild,sunny_mild,qp02z1_sunny,qp02z1_residential,qp02z1_mild
1,1,qp02zt,48,0:0,0.118507,residential,3,allowed,yes,31.104565,...,936.389375,0,0,0,-495.222,warm,sunny_warm,qp02zt_sunny,qp02zt_residential,qp02zt_warm
2,2,qp08bj,48,0:0,0.027132,residential,1,not allowed,no,25.919267,...,645.889151,0,0,0,-495.222,warm,sunny_warm,qp08bj_sunny,qp08bj_residential,qp08bj_warm
3,3,qp08gt,48,0:0,0.003272,residential,1,not allowed,no,16.405354,...,252.730296,0,0,0,-496.314,mild,rainy_mild,qp08gt_rainy,qp08gt_residential,qp08gt_mild
4,4,qp02zq,48,0:0,0.010819,residential,1,not allowed,no,10.803667,...,105.915563,0,0,0,-495.222,cold,rainy_cold,qp02zq_rainy,qp02zq_residential,qp02zq_cold


In [12]:
for col in train_df.columns:
    # print(col)
    # print(train_df[col].unique()[:5])
    print(col,": ",train_df[col].dtype)

Index :  int64
geohash :  str
day :  int64
timestamp :  str
demand :  float64
RoadType :  str
NumberofLanes :  int64
LargeVehicles :  str
Landmarks :  str
Temperature :  float64
Weather :  str
lat :  str
lon :  str
hour :  int64
geo_hour :  str
period :  str
geo_period :  str
roadtype_period :  str
sin_hour :  float64
cos_hour :  float64
temperature_sq :  float64
hour_sq :  int64
is_morning_peak :  int64
is_evening_peak :  int64
lat_lon_interaction :  float64
temp_bucket :  category
weather_temp :  str
geo_weather :  str
geo_road :  str
geo_temp :  str


# Training

In [13]:
import os
import joblib
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

In [14]:
TARGET = "demand"

DROP_COLS = [
    "Index",
    "timestamp",
    TARGET
]

FEATURE_COLS = [
    c for c in train_df.columns
    if c not in DROP_COLS
]

CAT_COLS = [
    "geohash",
    "RoadType",
    "LargeVehicles",
    "Landmarks",
    "Weather",
    "geo_hour",
    "period",
    "geo_period",
    "roadtype_period",

    # New categorical features
    "temp_bucket",
    "weather_temp",
    "geo_weather",
    "geo_road",
    "geo_temp",
]

In [15]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cat_scores = []

best_cat_model = None
best_cat_rmse = np.inf

In [16]:
for fold, (train_idx, valid_idx) in enumerate(kf.split(train_df), 1):

    print(f"\nFold {fold}")

    train_fold = train_df.iloc[train_idx].copy()
    valid_fold = train_df.iloc[valid_idx].copy()

    X_train = train_fold[FEATURE_COLS]
    y_train = train_fold[TARGET]

    X_valid = valid_fold[FEATURE_COLS]
    y_valid = valid_fold[TARGET]

    model = CatBoostRegressor(
        iterations=5000,
        learning_rate=0.015,
        depth=10,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42,
        early_stopping_rounds=300,
        verbose=100
    )

    model.fit(
        X_train,
        y_train,
        cat_features=CAT_COLS,
        eval_set=(X_valid, y_valid),
        use_best_model=True
    )

    preds = model.predict(X_valid)

    rmse = np.sqrt(
        mean_squared_error(
            y_valid,
            preds
        )
    )

    r2 = r2_score(
        y_valid,
        preds
    )

    cat_scores.append(rmse)

    print(f"Fold RMSE: {rmse:.6f}")
    print(f"Fold R²  : {r2:.6f}")

    if rmse < best_cat_rmse:
        best_cat_rmse = rmse
        best_cat_model = model


Fold 1
0:	learn: 0.1404026	test: 0.1404558	best: 0.1404558 (0)	total: 113ms	remaining: 9m 26s
100:	learn: 0.0558338	test: 0.0542709	best: 0.0542709 (100)	total: 4.87s	remaining: 3m 56s
200:	learn: 0.0412413	test: 0.0404579	best: 0.0404579 (200)	total: 9.85s	remaining: 3m 55s
300:	learn: 0.0372673	test: 0.0371151	best: 0.0371151 (300)	total: 15.6s	remaining: 4m 3s
400:	learn: 0.0354381	test: 0.0357010	best: 0.0357010 (400)	total: 21.8s	remaining: 4m 9s
500:	learn: 0.0341927	test: 0.0348565	best: 0.0348565 (500)	total: 27.3s	remaining: 4m 4s
600:	learn: 0.0330948	test: 0.0341515	best: 0.0341515 (600)	total: 32.7s	remaining: 3m 59s
700:	learn: 0.0322887	test: 0.0336870	best: 0.0336870 (700)	total: 37.9s	remaining: 3m 52s
800:	learn: 0.0315197	test: 0.0332374	best: 0.0332374 (800)	total: 43.3s	remaining: 3m 47s
900:	learn: 0.0308608	test: 0.0329035	best: 0.0329035 (900)	total: 48.7s	remaining: 3m 41s
1000:	learn: 0.0302832	test: 0.0326310	best: 0.0326310 (1000)	total: 54.1s	remaining: 3m 

In [17]:
cat_cv_rmse = np.mean(cat_scores)

print("\nCatBoost Results")
print("-" * 30)
print("Fold Scores:", cat_scores)
print("Mean RMSE:", cat_cv_rmse)


CatBoost Results
------------------------------
Fold Scores: [np.float64(0.030865348834991323), np.float64(0.031170198227856438), np.float64(0.030477521893019267), np.float64(0.031301751378637756), np.float64(0.030483711586605337)]
Mean RMSE: 0.030859706384222023


In [18]:
joblib.dump(
    best_cat_model,
    "models/best_model3.pkl"
)

joblib.dump(
    best_cat_model,
    "models/best_catboost3.pkl"
)

['models/best_catboost3.pkl']

In [19]:
import joblib
import pandas as pd

# Load model
model = joblib.load("models/best_catboost3.pkl")

# Feature importance
feature_importance = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance": model.get_feature_importance()
})

feature_importance = (
    feature_importance
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print(feature_importance)

                feature  importance
0              RoadType   30.590694
1               geohash   15.110093
2       roadtype_period   10.531422
3              geo_road    9.780796
4         NumberofLanes    5.005820
5              cos_hour    3.561590
6         LargeVehicles    3.059554
7            geo_period    3.012323
8   lat_lon_interaction    2.801335
9                   lon    2.592429
10             sin_hour    2.421346
11                 hour    1.606894
12              hour_sq    1.571948
13             geo_hour    1.384556
14                  day    1.286868
15                  lat    1.208766
16               period    1.164273
17             geo_temp    0.908853
18          geo_weather    0.652808
19         weather_temp    0.363307
20       temperature_sq    0.290716
21          Temperature    0.273448
22          temp_bucket    0.269323
23              Weather    0.238027
24            Landmarks    0.217417
25      is_morning_peak    0.054186
26      is_evening_peak    0

In [20]:
from sklearn.metrics import r2_score

# Training features
X_train_full = train_df[FEATURE_COLS]
y_train_full = train_df["demand"]

# Predictions on training data
train_preds = best_cat_model.predict(X_train_full)

# R² score
train_r2 = r2_score(y_train_full, train_preds)

print(f"Training R² Score: {train_r2:.6f}")

Training R² Score: 0.967139


In [21]:
def prepare_test_data(
    test_df
):
    """
    Complete preprocessing pipeline
    for inference.
    """

    df = test_df.copy()

    # cleaning
    df = clean_dataset(df)

    # feature engineering
    df = engineer_features(df)

    return df

In [22]:
test_processed = prepare_test_data(
    test_df
)

/tmp/ipykernel_15764/3791486193.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df.select_dtypes(include=["object"]).columns


In [23]:
test_processed.head()

,Index,geohash,day,timestamp,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,...,temperature_sq,hour_sq,is_morning_peak,is_evening_peak,lat_lon_interaction,temp_bucket,weather_temp,geo_weather,geo_road,geo_temp
0,0,qp02z1,49,2:15,residential,1,not allowed,no,16.457339,sunny,...,254.386654,4,0,0,-497.036,mild,sunny_mild,qp02z1_sunny,qp02z1_residential,qp02z1_mild
1,1,qp02z9,49,2:15,residential,1,not allowed,no,6.476213,snowy,...,35.465127,4,0,0,-497.036,cold,snowy_cold,qp02z9_snowy,qp02z9_residential,qp02z9_cold
2,2,qp02yf,49,2:15,residential,3,allowed,yes,22.318203,sunny,...,475.783964,4,0,0,-497.036,mild,sunny_mild,qp02yf_sunny,qp02yf_residential,qp02yf_mild
3,3,qp02z6,49,2:15,residential,2,not allowed,yes,16.457339,rainy,...,254.386654,4,0,0,-497.036,mild,rainy_mild,qp02z6_rainy,qp02z6_residential,qp02z6_mild
4,4,qp02zd,49,2:15,residential,1,not allowed,no,18.266162,foggy,...,315.386520,4,0,0,-497.036,mild,foggy_mild,qp02zd_foggy,qp02zd_residential,qp02zd_mild


In [24]:
X_test = test_processed[FEATURE_COLS]

test_preds = best_cat_model.predict(X_test)

In [25]:
import os
submission = pd.DataFrame({
    "Index": test_df["Index"],
    "demand": test_preds
})
os.makedirs(
    "../data/submissions",
    exist_ok=True
)

submission.to_csv(
    "../data/submissions/submission_4rthjune2.csv",
    index=False
)

print(submission.head())

   Index    demand
0      0  0.058944
1      1  0.020547
2      2  0.020058
3      3  0.026599
4      4  0.047562
